# Alcalá Voyages — AI Travel Planning Agent
**AAI-510: Agentic AI Systems | University of San Diego | June 2026**

**Team 6:**
- Paola Marsal — AI Engineer (Agent Architecture, Tools, Evaluation)
- Akua Duffour — Data Engineer (Data Pipeline)
- Corey Parker — Product Manager (Business Value, ROI)

---

## Overview
This notebook implements the Alcalá Voyages AI Travel Planning Agent using the **ReAct (Reasoning + Acting)** architecture pattern. The agent helps European travelers plan trips through a conversational interface by recommending destinations, providing climate information, and suggesting hotels grounded in real guest review data.

**Architecture:** ReAct single-agent with 4 custom tools
**LLMs Compared:** GPT-4o (OpenAI) vs Claude Sonnet 4 (Anthropic)
**Evaluation:** LLM-as-judge via MLflow with human-in-the-loop validation
**Dataset:** 69 European cities + 1,494 aggregated hotel records

**Tools:**
- `search_destinations()` — queries European cities by travel style and budget
- `get_climate_info()` — returns monthly temperature data by city
- `recommend_hotels()` — TF-IDF semantic search over 1,494 hotel records
- `reject_out_of_scope()` — gracefully declines irrelevant requests

## Step 1 — Install Dependencies
We pin specific library versions to ensure compatibility in the Databricks free-tier environment. Python is restarted after installation to load updated packages.

In [0]:
%pip install openai==1.30.0 anthropic==0.28.0 mlflow==2.13.0 scikit-learn pandas httpx==0.27.0
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.7/862.7 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: anthropic
    Found existing installation: anthropic 0.25.0
    Uninstalling anthropic-0.25.0:
      Successfully uninstalled anthropic-0.25.0
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Step 2 — Setup API Keys, Load Data, and Build TF-IDF Index
We load both cleaned Delta tables from the pipeline notebook, set API keys for GPT-4o and Claude Sonnet 4, and rebuild the TF-IDF search index on 1,494 aggregated hotel records. The index rebuilds each session since `/tmp/` files do not persist across cluster restarts.

In [0]:
import os
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# API Keys
os.environ["OPENAI_API_KEY"] = "DUMMY KEY" #removed API key to upload on Canvas
os.environ["ANTHROPIC_API_KEY"] = dbutils.secrets.get(scope="aai510", key="anthropic_api_key")

# Load cities data
cities_pd = spark.read.table("main.alcala_voyages.clean_europe_cities").toPandas()

# Load hotel aggregated data
hotels_pd = spark.read.table("main.alcala_voyages.clean_aggregated_hotels").toPandas()

# Rebuild TF-IDF index directly — no tmp files needed
print("Building TF-IDF index...")
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(
    hotels_pd['Hotel_Full_Description'].fillna('')
)

print(f"Cities loaded: {len(cities_pd)}")
print(f"Hotels loaded: {len(hotels_pd)}")
print(f"TF-IDF index built on {len(hotels_pd)} hotels!")

Building TF-IDF index...
Cities loaded: 69
Hotels loaded: 1494
TF-IDF index built on 1494 hotels!


## Step 2b — Agent Tool Definitions
The four tools form the agent's action space. Each tool is grounded in real data from the pipeline notebook. All tools include error handling to prevent agent crashes. The `reject_out_of_scope()` tool gracefully declines irrelevant requests, satisfying the rubric requirement for error handling.

**Tools:**
- `search_destinations()` — queries European cities by travel style (with style mapping), budget, and country
- `get_climate_info()` — parses JSON temperature data to return avg/high/low for a specific month
- `recommend_hotels()` — TF-IDF cosine similarity search over 1,494 aggregated hotel records
- `reject_out_of_scope()` — declines off-topic requests with helpful guidance on what the agent can do

In [0]:
import json
import pandas as pd

def search_destinations(travel_style: str = None, budget: str = None, country: str = None) -> str:
    """Search European cities by travel style, budget, or country."""
    try:
        results = cities_pd.copy()

        if country:
            results = results[results['country'].str.lower().str.contains(country.lower(), na=False)]

        if budget:
            results = results[results['budget_level'].str.lower().str.contains(budget.lower(), na=False)]

        if travel_style:
            # Map common travel style terms to dataset column names
            style_map = {
                "beach": "beaches", "beaches": "beaches",
                "culture": "culture", "cultural": "culture", "history": "culture", "historic": "culture",
                "adventure": "adventure", "outdoor": "adventure", "hiking": "adventure", "sport": "adventure",
                "nature": "nature", "scenic": "nature", "landscape": "nature",
                "nightlife": "nightlife", "party": "nightlife", "clubs": "nightlife",
                "food": "cuisine", "cuisine": "cuisine", "eating": "cuisine", "gastronomy": "cuisine",
                "wellness": "wellness", "spa": "wellness", "relax": "wellness",
                "urban": "urban", "city": "urban", "shopping": "urban",
                "seclusion": "seclusion", "quiet": "seclusion", "peaceful": "seclusion", "remote": "seclusion"
            }
            col = style_map.get(travel_style.lower())
            if col and col in results.columns:
                results = results.sort_values(col, ascending=False)
            else:
                # Try partial match on column names
                matching_cols = [c for c in results.columns 
                                if travel_style.lower() in c.lower()]
                if matching_cols:
                    results = results.sort_values(matching_cols[0], ascending=False)

        results = results.head(5)

        if results.empty:
            return f"No European destinations found matching your criteria. Try adjusting your search."

        output = []
        for _, row in results.iterrows():
            desc = str(row.get('short_description', ''))[:100] + "..." if row.get('short_description') else ""
            output.append(
                f"- {row['city']}, {row['country']} | Budget: {row.get('budget_level', 'N/A')} | {desc}"
            )

        return "Top destination recommendations:\n" + "\n".join(output)

    except Exception as e:
        return f"Destination search encountered an error: {str(e)}. Please try a different query."


def get_climate_info(city: str, month: str = None) -> str:
    """Get climate information for a European city, optionally filtered by month."""
    try:
        result = cities_pd[cities_pd['city'].str.lower().str.contains(city.lower(), na=False)]

        if result.empty:
            return f"No climate data found for {city}. Please check the city name and try again."

        row = result.iloc[0]
        city_name = row['city']
        country = row['country']

        # Month name to number mapping
        month_map = {
            "january": "1", "february": "2", "march": "3", "april": "4",
            "may": "5", "june": "6", "july": "7", "august": "8",
            "september": "9", "october": "10", "november": "11", "december": "12",
            "jan": "1", "feb": "2", "mar": "3", "apr": "4", "jun": "6",
            "jul": "7", "aug": "8", "sep": "9", "oct": "10", "nov": "11", "dec": "12"
        }

        temp_data = row.get('avg_temp_monthly')

        if temp_data and pd.notna(temp_data):
            try:
                # Parse JSON temperature data
                if isinstance(temp_data, str):
                    temps = json.loads(temp_data)
                else:
                    temps = temp_data

                if month:
                    # Get specific month
                    month_num = month_map.get(month.lower().strip())
                    if month_num and month_num in temps:
                        m = temps[month_num]
                        return (
                            f"Climate in {city_name}, {country} in {month.capitalize()}:\n"
                            f"- Average temperature: {m.get('avg', 'N/A')}°C\n"
                            f"- High: {m.get('max', 'N/A')}°C\n"
                            f"- Low: {m.get('min', 'N/A')}°C"
                        )
                    else:
                        return f"Month '{month}' not recognized. Please use a full month name like 'July'."
                else:
                    # Return summary of best months (warmest 3)
                    monthly_avgs = {k: v.get('avg', 0) for k, v in temps.items()}
                    top_months = sorted(monthly_avgs.items(), key=lambda x: float(x[1]), reverse=True)[:3]
                    month_names = {
                        "1": "January", "2": "February", "3": "March", "4": "April",
                        "5": "May", "6": "June", "7": "July", "8": "August",
                        "9": "September", "10": "October", "11": "November", "12": "December"
                    }
                    best = ", ".join([f"{month_names.get(m, m)} ({avg}°C avg)" 
                                     for m, avg in top_months])
                    return (
                        f"Climate in {city_name}, {country}:\n"
                        f"- Best months to visit: {best}\n"
                        f"- Ask about a specific month for detailed temperature data."
                    )

            except (json.JSONDecodeError, TypeError, ValueError) as e:
                return f"{city_name}, {country} is a European destination. Climate data parsing error: {str(e)}"
        else:
            return f"{city_name}, {country} is in Europe. Best travel months are generally April through October."

    except Exception as e:
        return f"Climate lookup encountered an error: {str(e)}. Please try again."


def recommend_hotels(query: str, top_n: int = 3) -> str:
    """Find hotels using TF-IDF semantic search on guest reviews."""
    try:
        if not query or len(query.strip()) == 0:
            return "Please provide hotel preferences to search for recommendations."

        query_vec = vectorizer.transform([query])
        scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
        top_indices = scores.argsort()[-top_n:][::-1]

        results = hotels_pd.iloc[top_indices]

        if results.empty:
            return "No hotels found matching your preferences. Try different search terms."

        output = []
        for _, row in results.iterrows():
            score = round(row.get('Final_Average_Score', 0), 1)
            name = row.get('Hotel_Name', 'Unknown Hotel')
            address = row.get('Hotel_Address', 'Address not available')
            output.append(f"- {name} | {address} | Rating: {score}/10")

        return "Recommended hotels based on your preferences:\n" + "\n".join(output)

    except Exception as e:
        return f"Hotel search encountered an error: {str(e)}. Please try different search terms."


def reject_out_of_scope(reason: str = "") -> str:
    """Gracefully decline requests outside the travel planning scope."""
    return (
        "I'm sorry, that request is outside my area of expertise. "
        "I am Alcalá Voyages' European travel planning assistant. "
        "I can help you with:\n"
        "- European destination recommendations\n"
        "- Climate and weather information by city and month\n"
        "- Hotel suggestions based on your preferences\n"
        "Please ask me about planning your European trip!"
    )


# Tool registry
TOOLS = {
    "search_destinations": search_destinations,
    "get_climate_info": get_climate_info,
    "recommend_hotels": recommend_hotels,
    "reject_out_of_scope": reject_out_of_scope
}

print("All 4 tools defined successfully!")
print("- search_destinations: travel style mapping + error handling")
print("- get_climate_info: JSON parsing + month lookup + error handling")
print("- recommend_hotels: TF-IDF search + error handling")
print("- reject_out_of_scope: graceful rejection with guidance")

All 4 tools defined successfully!
- search_destinations: travel style mapping + error handling
- get_climate_info: JSON parsing + month lookup + error handling
- recommend_hotels: TF-IDF search + error handling
- reject_out_of_scope: graceful rejection with guidance


## Step 3 — ReAct Agent Implementation
Two agent functions implement the ReAct loop — one for GPT-4o and one for Claude Sonnet 4. Each function follows the same pattern: check scope, reason, invoke tools, observe results, synthesize response. This enables a direct performance comparison on identical queries.

In [0]:
import openai
import anthropic
import mlflow
import json
import re

# Out of scope keywords
OUT_OF_SCOPE_KEYWORDS = [
    "flight", "visa", "medical", "insurance", "stock", "weather forecast",
    "restaurant reservation", "car rental", "cruise", "train ticket"
]

def is_out_of_scope(query: str) -> bool:
    return any(kw in query.lower() for kw in OUT_OF_SCOPE_KEYWORDS)

# Tool definitions for LLM function calling
tool_definitions = [
    {
        "type": "function",
        "function": {
            "name": "search_destinations",
            "description": "Search European cities by travel style, budget level, or country",
            "parameters": {
                "type": "object",
                "properties": {
                    "travel_style": {"type": "string", "description": "Travel style e.g. beaches, culture, adventure"},
                    "budget": {"type": "string", "description": "Budget level: Luxury, Mid-range, or Budget"},
                    "country": {"type": "string", "description": "European country name"}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_climate_info",
            "description": "Get climate and temperature information for a European city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "month": {"type": "string", "description": "Month of travel"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "recommend_hotels",
            "description": "Recommend hotels based on traveler preferences using guest review search",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Hotel preferences e.g. romantic, family friendly, central location"},
                    "top_n": {"type": "integer", "description": "Number of hotels to return"}
                },
                "required": ["query"]
            }
        }
    }
]

def run_agent_gpt4o(user_query: str) -> dict:
    """Run the ReAct agent using GPT-4o."""
    
    # Check out of scope first
    if is_out_of_scope(user_query):
        return {
            "llm": "GPT-4o",
            "query": user_query,
            "response": reject_out_of_scope(user_query),
            "tools_called": ["reject_out_of_scope"],
            "out_of_scope": True
        }
    
    client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    
    messages = [
        {"role": "system", "content": """You are the Alcalá Voyages AI travel planning assistant, 
        specializing in European travel. Help users plan their European trips by recommending 
        destinations, providing climate information, and suggesting hotels. 
        Always use the available tools to provide grounded recommendations.
        Be friendly, specific, and helpful."""},
        {"role": "user", "content": user_query}
    ]
    
    tools_called = []
    max_iterations = 5
    iteration = 0
    
    while iteration < max_iterations:
        iteration += 1
        
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tool_definitions,
            tool_choice="auto"
        )
        
        message = response.choices[0].message
        
        # If no tool calls, we have our final answer
        if not message.tool_calls:
            return {
                "llm": "GPT-4o",
                "query": user_query,
                "response": message.content,
                "tools_called": tools_called,
                "out_of_scope": False
            }
        
        # Process tool calls
        messages.append(message)
        
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            tools_called.append(tool_name)
            
            # Execute the tool
            if tool_name in TOOLS:
                tool_result = TOOLS[tool_name](**tool_args)
            else:
                tool_result = f"Tool {tool_name} not found"
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            })
    
    return {
        "llm": "GPT-4o",
        "query": user_query,
        "response": "Max iterations reached",
        "tools_called": tools_called,
        "out_of_scope": False
    }


def run_agent_claude(user_query: str) -> dict:
    """Run the ReAct agent using Claude Sonnet 4."""
    
    # Check out of scope first
    if is_out_of_scope(user_query):
        return {
            "llm": "Claude Sonnet 4",
            "query": user_query,
            "response": reject_out_of_scope(user_query),
            "tools_called": ["reject_out_of_scope"],
            "out_of_scope": True
        }
    
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    
    # Claude tool format
    claude_tools = [
        {
            "name": "search_destinations",
            "description": "Search European cities by travel style, budget level, or country",
            "input_schema": {
                "type": "object",
                "properties": {
                    "travel_style": {"type": "string", "description": "Travel style e.g. beaches, culture, adventure"},
                    "budget": {"type": "string", "description": "Budget level: Luxury, Mid-range, or Budget"},
                    "country": {"type": "string", "description": "European country name"}
                }
            }
        },
        {
            "name": "get_climate_info",
            "description": "Get climate and temperature information for a European city",
            "input_schema": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "month": {"type": "string", "description": "Month of travel"}
                },
                "required": ["city"]
            }
        },
        {
            "name": "recommend_hotels",
            "description": "Recommend hotels based on traveler preferences using guest review search",
            "input_schema": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Hotel preferences"},
                    "top_n": {"type": "integer", "description": "Number of hotels to return"}
                },
                "required": ["query"]
            }
        }
    ]
    
    messages = [{"role": "user", "content": user_query}]
    tools_called = []
    max_iterations = 5
    iteration = 0
    
    while iteration < max_iterations:
        iteration += 1
        
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=2048,
            system="""You are the Alcalá Voyages AI travel planning assistant, 
            specializing in European travel. Help users plan their European trips by recommending 
            destinations, providing climate information, and suggesting hotels. 
            Always use the available tools to provide grounded recommendations.
            Be friendly, specific, and helpful.""",
            messages=messages,
            tools=claude_tools
        )
        
        # Check if done
        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, 'text'):
                    final_text += block.text
            return {
                "llm": "Claude Sonnet 4",
                "query": user_query,
                "response": final_text,
                "tools_called": tools_called,
                "out_of_scope": False
            }
        
        # Process tool use
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                tool_name = block.name
                tool_args = block.input
                tools_called.append(tool_name)
                
                if tool_name in TOOLS:
                    tool_result = TOOLS[tool_name](**tool_args)
                else:
                    tool_result = f"Tool {tool_name} not found"
                
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": tool_result
                })
        
        # Add assistant response and tool results to messages
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
    
    return {
        "llm": "Claude Sonnet 4",
        "query": user_query,
        "response": "Max iterations reached",
        "tools_called": tools_called,
        "out_of_scope": False
    }

print("Agent functions defined successfully!")

Agent functions defined successfully!


## Step 4 — LLM-as-Judge Evaluation
GPT-4o serves as the evaluation judge, scoring each agent response on three dimensions on a 1-5 scale: relevance, accuracy, and helpfulness. This automated evaluation is validated by human review of all traces.

In [0]:
def llm_judge(query: str, response: str, llm_name: str) -> dict:
    """Score agent response using GPT-4o as judge."""
    
    client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    
    judge_prompt = f"""You are an expert evaluator for an AI travel planning assistant.
    
User Query: {query}
Agent Response ({llm_name}): {response}

Score this response on three dimensions from 1-5:
1. Relevance: Does the response directly address the user's travel query?
2. Accuracy: Are the recommendations grounded and specific (not vague)?
3. Helpfulness: Would a real traveler find this response useful and actionable?

Respond in this exact JSON format:
{{
    "relevance": <1-5>,
    "accuracy": <1-5>,
    "helpfulness": <1-5>,
    "rationale": "<brief explanation>"
}}"""

    judge_response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": judge_prompt}],
        temperature=0
    )
    
    try:
        import json
        content = judge_response.choices[0].message.content
        # Strip markdown if present
        content = content.replace("```json", "").replace("```", "").strip()
        scores = json.loads(content)
    except:
        scores = {"relevance": 0, "accuracy": 0, "helpfulness": 0, "rationale": "Parse error"}
    
    return scores

print("LLM judge defined!")

LLM judge defined!


## Step 5 — Evaluation Traces (MLflow)
Five evaluation traces are logged to MLflow. Both GPT-4o and Claude Sonnet 4 are run on every trace for a complete multi-LLM comparison. Trace 4 demonstrates graceful out-of-scope rejection. All traces are scored by an LLM-as-judge on relevance, accuracy, and helpfulness (1-5 scale). All 5 traces were also manually reviewed by the team for human-in-the-loop validation.

**Test Queries:**
1. Cultural city in Spain — mid-range budget
2. Paris climate in July
3. Romantic hotel in London
4. Flight booking request — out-of-scope rejection
5. Adventure in Austria — multi-LLM comparison

In [0]:
import mlflow
import json

mlflow.set_experiment("/Shared/alcala_voyages_agent_team6")

# Define 5 test queries
test_queries = [
    "I want to visit a cultural city in Spain with a mid-range budget. What do you recommend?",
    "What is the climate like in Paris in July?",
    "I need a romantic hotel in London with great reviews.",
    "Can you help me book a flight to Rome?",
    "I want adventure activities in Austria. What cities and hotels do you suggest?"
]

all_results = []

for i, query in enumerate(test_queries):
    print(f"\n{'='*60}")
    print(f"TRACE {i+1}: {query}")
    print('='*60)

    with mlflow.start_run(run_name=f"trace_{i+1}"):

        mlflow.log_param("query", query)
        mlflow.log_param("trace_number", i+1)

        # Run GPT-4o
        gpt_result = run_agent_gpt4o(query)
        print(f"\nGPT-4o Response:\n{gpt_result['response']}")
        print(f"Tools called: {gpt_result['tools_called']}")

        # Judge GPT-4o
        gpt_scores = llm_judge(query, gpt_result['response'], "GPT-4o")
        gpt_avg = (gpt_scores.get('relevance',0) + gpt_scores.get('accuracy',0) + gpt_scores.get('helpfulness',0)) / 3

        mlflow.log_param("llm_gpt4o", "gpt-4o")
        mlflow.log_metric("gpt4o_relevance", gpt_scores.get('relevance', 0))
        mlflow.log_metric("gpt4o_accuracy", gpt_scores.get('accuracy', 0))
        mlflow.log_metric("gpt4o_helpfulness", gpt_scores.get('helpfulness', 0))
        mlflow.log_metric("gpt4o_avg_score", gpt_avg)
        print(f"\nGPT-4o Judge Scores: {gpt_scores}")

        # Run Claude Sonnet 4 on ALL traces
        print(f"\nRunning Claude Sonnet 4...")
        claude_result = run_agent_claude(query)
        print(f"\nClaude Sonnet 4 Response:\n{claude_result['response']}")
        print(f"Tools called: {claude_result['tools_called']}")

        # Judge Claude
        claude_scores = llm_judge(query, claude_result['response'], "Claude Sonnet 4")
        claude_avg = (claude_scores.get('relevance',0) + claude_scores.get('accuracy',0) + claude_scores.get('helpfulness',0)) / 3

        mlflow.log_param("llm_claude", "claude-sonnet-4-5")
        mlflow.log_metric("claude_relevance", claude_scores.get('relevance', 0))
        mlflow.log_metric("claude_accuracy", claude_scores.get('accuracy', 0))
        mlflow.log_metric("claude_helpfulness", claude_scores.get('helpfulness', 0))
        mlflow.log_metric("claude_avg_score", claude_avg)
        print(f"\nClaude Judge Scores: {claude_scores}")

        print(f"\n--- COMPARISON ---")
        print(f"GPT-4o avg:        {round(gpt_avg, 2)}")
        print(f"Claude Sonnet avg: {round(claude_avg, 2)}")

        all_results.append({
            "trace": i+1,
            "query": query,
            "gpt4o_response": gpt_result['response'],
            "gpt4o_scores": gpt_scores,
            "gpt4o_avg": round(gpt_avg, 2),
            "claude_response": claude_result['response'],
            "claude_scores": claude_scores,
            "claude_avg": round(claude_avg, 2)
        })

# Print final comparison summary
print("\n\n" + "="*60)
print("FINAL LLM COMPARISON SUMMARY")
print("="*60)
total_gpt = sum(r['gpt4o_avg'] for r in all_results) / len(all_results)
total_claude = sum(r['claude_avg'] for r in all_results) / len(all_results)
print(f"GPT-4o overall avg:        {round(total_gpt, 2)}/5")
print(f"Claude Sonnet 4 overall avg: {round(total_claude, 2)}/5")
print(f"Cost per query (GPT-4o):   ~$0.005")
print(f"Cost per query (Claude):   ~$0.003")
print(f"Performance delta:         {round(abs(total_gpt - total_claude), 2)}")
print("="*60)
print("\nAll 5 traces completed and logged to MLflow!")


TRACE 1: I want to visit a cultural city in Spain with a mid-range budget. What do you recommend?

GPT-4o Response:
For a cultural trip in Spain on a mid-range budget, I recommend considering these cities:

1. **Barcelona**
   - Experience Gaudí's whimsical architecture, lively tapas bars, and the warm Mediterranean lifestyle.

2. **Madrid**
   - Explore vibrant streets filled with lively tapas bars, stunning architecture, and a rich tapestry of history and art.

3. **Seville**
   - Enjoy sun-drenched plazas, the aroma of orange blossoms, and the passionate rhythm of flamenco that create an enchanting ambiance.

4. **Bilbao**
   - Discover a vibrant blend of modern architecture and rich history, bustling markets, and delightful pintxos bars.

5. **Granada**
   - Stroll through winding cobblestone streets to vibrant plazas filled with the aromas of tapas, with the Alhambra's majesty as a backdrop.

Would you like climate information or hotel recommendations for any of these cities?
Too

## Step 6 — Agent Performance Commentary
Human-in-the-loop validation: All 5 traces were manually reviewed by the team. Key finding from human review: Trace 4 (out-of-scope rejection) received low LLM judge scores because the judge evaluated helpfulness from a user perspective rather than agent correctness. Human review is essential for catching these edge cases.

In [0]:
print("""
=============================================================
AGENT PERFORMANCE COMMENTARY — ALCALÁ VOYAGES TRAVEL AGENT
=============================================================

OVERVIEW:
The Alcalá Voyages ReAct agent was evaluated across 5 traces
using both GPT-4o and Claude Sonnet 4 on every trace.
The agent uses 4 tools grounded in real datasets.

TRACE RESULTS:
- Trace 1 (Spain destinations): Both LLMs scored 5.0/5.
  search_destinations() returned relevant mid-range cities
  with descriptions from the dataset.
- Trace 2 (Paris climate): Both LLMs scored 5.0/5.
  get_climate_info() correctly parsed JSON temperature data
  and returned avg/high/low for July specifically.
- Trace 3 (London hotel): GPT-4o 4.33/5, Claude 4.67/5.
  Claude returned 5 hotels vs GPT-4o returning 1.
  recommend_hotels() used TF-IDF RAG on 1,494 hotels.
- Trace 4 (Flight booking): Both scored 2.33/5.
  Both correctly rejected as out-of-scope.
  Low score reflects judge evaluating from user perspective
  not agent correctness. Human review confirms rejection
  was appropriate and working as designed.
- Trace 5 (Austria adventure): GPT-4o 2.33/5, Claude 3.33/5.
  Claude provided better structured response with top pick
  recommendation. Both limited by hotel dataset skewing
  toward Vienna rather than Innsbruck/Hallstatt.

MULTI-LLM COMPARISON (All 5 Traces):
- GPT-4o overall average:         3.80/5
- Claude Sonnet 4 overall avg:    4.07/5
- Claude outperformed GPT-4o by 0.27 points across all traces.
- Claude won on Traces 3 and 5 by providing richer responses,
  more hotel options, better formatting, and stronger context.
- Both LLMs tied on Traces 1, 2, and 4.

ROI ANALYSIS:
- GPT-4o: ~$0.005 per query
- Claude Sonnet 4: ~$0.003 per query (40% cheaper)
- Performance delta: Claude scored HIGHER by 0.27 points
- At 500 clients/month: Claude saves ~$12/year in API costs
  while delivering measurably better responses
- Recommendation: Claude Sonnet 4 is the clear winner —
  better performance at lower cost for Alcalá Voyages.

HUMAN IN THE LOOP:
All 5 traces were manually reviewed by the team:
1. Trace 4 low scores do not reflect agent correctness.
   The out-of-scope rejection was appropriate. Automated
   judges penalize scope-limiting behavior that is actually
   correct from an architectural standpoint.
2. Claude provided richer hotel results on Traces 3 and 5,
   explaining its higher scores on those traces.
3. Both models struggled with hotel geography on Trace 5.
   The hotel dataset skews toward Vienna — this is a data
   limitation, not a model limitation. Future improvement
   would add hotel data for Innsbruck and Hallstatt.

AGENT STRENGTHS:
- Climate data highly accurate — JSON parsing returns real
  avg/high/low temperatures for specific months requested
- Destination search improved with travel style mapping
  (beach, cultural, hiking all map to correct columns)
- Hotel RAG returned specific named properties with ratings
- Out-of-scope detection working correctly on Trace 4
- Claude Sonnet 4 consistently provided richer responses
- All tools have error handling — no crashes during evaluation

AGENT LIMITATIONS:
- Hotel index skews toward Vienna, limiting diversity
- TF-IDF less powerful than true vector embeddings
- Dataset limited to 6 European countries
- Adventure-specific hotel filtering could be improved
- Out-of-scope uses keyword pre-check rather than LLM judgment

WHAT WE LEARNED:
Building a production-grade agent requires careful attention
to data quality. The hotel dataset geographic skew directly
impacted recommendation quality for non-Vienna queries.
LLM-as-judge evaluation is powerful but requires human
validation, especially for out-of-scope handling edge cases.
Claude Sonnet 4 is the stronger choice for travel planning
agents — it matched or exceeded GPT-4o on every trace at
40% lower cost, making it the clear ROI winner.
============================================================
""")


AGENT PERFORMANCE COMMENTARY — ALCALÁ VOYAGES TRAVEL AGENT

OVERVIEW:
The Alcalá Voyages ReAct agent was evaluated across 5 traces
using both GPT-4o and Claude Sonnet 4 on every trace.
The agent uses 4 tools grounded in real datasets.

TRACE RESULTS:
- Trace 1 (Spain destinations): Both LLMs scored 5.0/5.
  search_destinations() returned relevant mid-range cities
  with descriptions from the dataset.
- Trace 2 (Paris climate): Both LLMs scored 5.0/5.
  get_climate_info() correctly parsed JSON temperature data
  and returned avg/high/low for July specifically.
- Trace 3 (London hotel): GPT-4o 4.33/5, Claude 4.67/5.
  Claude returned 5 hotels vs GPT-4o returning 1.
  recommend_hotels() used TF-IDF RAG on 1,494 hotels.
- Trace 4 (Flight booking): Both scored 2.33/5.
  Both correctly rejected as out-of-scope.
  Low score reflects judge evaluating from user perspective
  not agent correctness. Human review confirms rejection
  was appropriate and working as designed.
- Trace 5 (Austria adv